# Multi-Source Information Retrieval & Confidence Scoring Assignment

## Task Overview
You are given:
- A CSV file (`claims.csv`) containing **fictional entities** and a type of claim to verify.
- A folder of HTML pages (`webpages/`) containing the information needed to answer those claims.

Your job:
1. Implement the function `extract_answer(entity, claim_type, webpages)`.
2. For **each claim**, find the correct value from the webpages.
3. Assign a confidence score (0–100) for your answer.
4. Provide a short evidence snippet from the text supporting your answer.

**IMPORTANT:**
- Do NOT rename or remove the provided function — the evaluation system depends on it.
- Your script must output a file named `results.csv` with **exactly these columns**:
    - `id` — The claim ID from the CSV.
    - `found_value` — Your extracted answer (string or number).
    - `confidence_score` — Integer between 0 and 100.
    - `evidence_snippet` — A short snippet of text from the webpages showing the answer.

**Evaluation:**
Your output will be graded automatically against a hidden ground truth.
Scoring is based on:
- Accuracy of `found_value`
- Calibration of `confidence_score`
- Correctness of `evidence_snippet`

**Time estimate:** 3 days

---


In [ ]:
import pandas as pd
from pathlib import Path

# Load the claims CSV (no expected_value column here)
claims_df = pd.read_csv("claims.csv")

print(f"Loaded {len(claims_df)} claims")
claims_df.head()


In [ ]:
# Load all HTML pages into a dictionary {filename: content}
webpages_dir = Path("webpages")
webpages = {}

for html_file in webpages_dir.glob("*.html"):
    with open(html_file, "r", encoding="utf-8") as f:
        webpages[html_file.name] = f.read()

print(f"Loaded {len(webpages)} webpages")
list(webpages.keys())[:5]


In [ ]:
import re
from typing import Tuple, Optional, List


def extract_answer(entity, claim_type, webpages):
    """Find the value for a (entity, claim_type) by parsing the webpages.

    Parameters:
        entity (str): Name of the fictional entity (e.g., 'Entity_042').
        claim_type (str): Type of claim (e.g., 'Annual Revenue (USD Millions)').
        webpages (dict): Mapping of filename -> HTML content.

    Returns:
        tuple: (found_value, confidence_score, evidence_snippet)
    """

    # Normalize helpers
    def normalize_text(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def to_lower_no_extra_ws(text: str) -> str:
        return re.sub(r"\s+", " ", text.lower())

    def cast_if_numeric(value_text: str, claim_label: str):
        cleaned = value_text.strip()
        # Remove thousands separators if any
        cleaned_digits = cleaned.replace(",", "")
        # Decide if this claim is numeric/year-like
        numeric_claims = {
            "number of offices",
            "annual revenue (usd millions)",
            "subscription price (usd)",
            "customer satisfaction (score)",
            "employee count (approx)",
            "public listing year",
            "founded year",
            "product launch year",
            "last funding round year",
        }
        if claim_label in numeric_claims:
            # Extract the first integer in the text
            m = re.search(r"(-?\d+)", cleaned_digits)
            if m:
                try:
                    return int(m.group(1))
                except ValueError:
                    pass
        return cleaned

    # Map claim types to canonical field phrases observed in webpages
    canonical_field_aliases = {
        "number of offices": ["number of offices"],
        "main competitor": ["main competitor"],
        "annual revenue (usd millions)": ["annual revenue (usd millions)"],
        "research lab location": ["research lab location"],
        "ceo": ["ceo"],
        "subscription price (usd)": ["subscription price (usd)"],
        "support email": ["support email"],
        "recent award": ["recent award"],
        "regional hq": ["regional hq", "headquarters"],
        "headquarters neighborhood": ["headquarters neighborhood"],
        "manufacturing site": ["manufacturing site"],
        "public listing year": ["public listing year"],
        "core technology": ["core technology"],
        "major client": ["major client"],
        "founded year": ["founded year"],
        "partner organization": ["partner organization"],
        "product launch year": ["product launch year", "launch year"],
        "customer satisfaction (score)": ["customer satisfaction (score)"],
        "employee count (approx)": ["employee count (approx)"],
        "primary market": ["primary market"],
        "flagship product": ["flagship product"],
        "last funding round year": ["last funding round year"],
        "founder": ["founder"],
        "slogan": ["slogan"],
    }

    entity_escaped = re.escape(entity)

    claim_norm = claim_type.strip().lower()
    field_aliases: List[str] = canonical_field_aliases.get(claim_norm, [claim_norm])

    # Build regex patterns for different phrasings. We will search sentence-level.
    # We keep patterns simple and robust across noise.
    def build_patterns(field_phrase: str) -> List[re.Pattern]:
        # Common value group: allow emails, words, spaces, dots, dashes, apostrophes, and numbers
        val = r"(?P<val>[A-Za-z0-9@._%+\- '&/]+)"
        # Patterns where the entity is referenced directly
        patterns = [
            # "Entity_X reports that its FIELD is VALUE"
            rf"{entity_escaped}[^.?!]*?\b{re.escape(field_phrase)}\b[^.?!]*?\bis\s+{val}",
            # "Entity_X with FIELD equal to VALUE"
            rf"{entity_escaped}[^.?!]*?\b{re.escape(field_phrase)}\b[^.?!]*?\bequal to\s+{val}",
            # "Entity_X had FIELD listed as VALUE"
            rf"{entity_escaped}[^.?!]*?\b{re.escape(field_phrase)}\b[^.?!]*?\blisted as\s+{val}",
            # "Entity_X's FIELD as VALUE"
            rf"{entity_escaped}[^.?!]*?'s\s+{re.escape(field_phrase)}\s+as\s+{val}",
        ]
        # Special phrasing for HQ
        if field_phrase in {"headquarters", "regional hq"}:
            patterns.extend([
                rf"{entity_escaped}[^.?!]*?is\s+headquartered\s+in\s+{val}",
                rf"{entity_escaped}[^.?!]*?\bheadquarters\b[^.?!]*?\bas\s+{val}",
            ])
        # Special phrasing for Founded Year (sometimes "founded in")
        if field_phrase in {"founded year"}:
            patterns.extend([
                rf"{entity_escaped}[^.?!]*?\bfounded\s+in\s+(?P<val>\d{{4}})",
            ])
        return [re.compile(p, flags=re.IGNORECASE | re.DOTALL) for p in patterns]

    # Attempt extraction by scanning all webpages and their sentences
    best_match: Optional[Tuple[str, int, str]] = None

    for filename, content in webpages.items():
        original = content
        lowered = content.lower()

        # Quick filter: if entity not mentioned, skip
        if entity.lower() not in lowered:
            continue

        # Search across all alias phrasings
        for alias in field_aliases:
            for pattern in build_patterns(alias):
                m = pattern.search(content)
                if not m:
                    continue
                raw_value = m.group("val")
                # Clean value: trim trailing punctuation
                value_clean = normalize_text(re.sub(r"[\s\.,;:]+$", "", raw_value))

                # Evidence snippet: show ~160 chars around match
                start, end = m.span()
                snippet_start = max(0, start - 80)
                snippet_end = min(len(original), end + 80)
                snippet = normalize_text(original[snippet_start:snippet_end])

                # Heuristic confidence based on phrasing strength
                pattern_text = pattern.pattern.lower()
                if any(kw in pattern_text for kw in ["equal to", "listed as", "'s", " is "]):
                    confidence = 95
                else:
                    confidence = 90

                # Post-process numeric casting when relevant
                value_final = cast_if_numeric(value_clean, claim_norm)

                best_match = (value_final, confidence, snippet)
                # Return on first strong match
                return best_match

    # If no pattern matched, try a very loose fallback for HQ phrasing only
    if claim_norm in {"regional hq"}:
        hq_pat = re.compile(rf"{entity_escaped}[^.?!]*?headquartered\s+in\s+(?P<val>[A-Za-z0-9 .\-'/&]+)", re.IGNORECASE | re.DOTALL)
        for filename, content in webpages.items():
            if entity.lower() not in content.lower():
                continue
            m = hq_pat.search(content)
            if m:
                raw_value = m.group("val")
                value_clean = normalize_text(re.sub(r"[\s\.,;:]+$", "", raw_value))
                start, end = m.span()
                snippet_start = max(0, start - 80)
                snippet_end = min(len(content), end + 80)
                snippet = normalize_text(content[snippet_start:snippet_end])
                return (value_clean, 88, snippet)

    # Default if not found
    return "UNKNOWN", 0, "No evidence found"


In [ ]:
# Prefer the extractor from run_extractor.py for better coverage
try:
    from run_extractor import extract_answer as _extract_answer_from_script
    extract_answer = _extract_answer_from_script
    print("Using extract_answer from run_extractor.py")
except Exception as e:
    print("Falling back to notebook extract_answer due to import error:", e)


In [ ]:
results = []

for _, row in claims_df.iterrows():
    claim_id = row["id"]
    entity = row["entity"]
    claim_type = row["claim_type"]

    found_value, confidence_score, evidence_snippet = extract_answer(entity, claim_type, webpages)

    results.append({
        "id": claim_id,
        "found_value": found_value,
        "confidence_score": confidence_score,
        "evidence_snippet": evidence_snippet
    })

results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)

# Also save a submission-named copy
first_name = "Kunal"
results_df.to_csv(f"{first_name}_results.csv", index=False)

print("Saved results.csv with", len(results_df), "rows")
results_df.head()


## Submission
- Submit **both** your completed `.ipynb` notebook **and** the `results.csv` file.
- **IMPORTANT:** Name your results file exactly as follows:  
  `[FirstName]_results.csv`  
  Example: `John_results.csv`
- Do not modify the output CSV format or column names.
- Ensure your code runs from start to finish without errors.

Good luck!
